In [ ]:
!pip install pennylane

In [ ]:
# ============================================================
# NOTEBOOK A: CLASSICAL BiLSTM — LARGE-SCALE BASELINE (FINAL)
# ============================================================

import time
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ============================================================
# Reproducibility
# ============================================================
SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

# ============================================================
# Hyperparameters
# ============================================================
MAX_SAMPLES = 500_000
VOCAB_SIZE  = 10_000
MAX_LEN     = 50
BATCH_SIZE  = 128
EPOCHS      = 20
LR          = 1e-4
N_RUNS      = 5

# ============================================================
# Learning-rate Scheduler
# ============================================================
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# ============================================================
# Dataset Loader
# ============================================================
def load_sentiment140(path):
    cols = ["target", "id", "date", "query", "user", "text"]
    df = pd.read_csv(path, encoding="latin-1", header=None, names=cols)
    df = df[["target", "text"]]
    df["target"] = df["target"].replace({4: 1})
    return df

FILE_PATH = "/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv"
df = load_sentiment140(FILE_PATH)

df = (
    df.groupby("target", group_keys=False)
      .apply(lambda x: x.sample(MAX_SAMPLES // 2, random_state=SEED))
      .sample(frac=1.0, random_state=SEED)
      .reset_index(drop=True)
)

print(f"[INFO] Dataset size used: {len(df)}")

X = df.text.values
y = df.target.values

# ============================================================
# Train / Val / Test Split (60 / 20 / 20)
# ============================================================
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED
)

print(f"[INFO] Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# ============================================================
# Tokenizer (TRAIN ONLY)
# ============================================================
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

def encode(texts):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

X_train = encode(X_train)
X_val   = encode(X_val)
X_test  = encode(X_test)

# ============================================================
# Model Definition
# ============================================================
def build_model():
    model = Sequential([
        Embedding(VOCAB_SIZE, 128),
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.5),
        LSTM(64),
        Dense(64, activation="relu"),
        Dense(2, activation="softmax")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ============================================================
# Training Runs
# ============================================================
all_metrics = []
epoch_counts = []
histories = []

for run in range(N_RUNS):
    print(f"\n==============================")
    print(f" RUN {run+1}/{N_RUNS}")
    print("==============================")

    tf.keras.utils.set_random_seed(run)
    model = build_model()

    start = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            EarlyStopping(patience=3, restore_best_weights=True),
            lr_scheduler
        ],
        verbose=1
    )
    train_time = time.time() - start

    epochs_used = len(history.history["loss"])
    epoch_counts.append(epochs_used)
    histories.append(history.history)

    print(f"[INFO] Training stopped at epoch: {epochs_used}")   # <<< ADDED

    # ---------------------
    # Test Evaluation
    # ---------------------
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)

    print(f"Precision: {prec:.4f}")   # <<< ADDED
    print(f"Recall   : {rec:.4f}")    # <<< ADDED

    all_metrics.append({
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": prec,
        "recall": rec,
        "f1": f1_score(y_test, y_pred),
        "train_time_sec": train_time,
        "epochs_used": epochs_used
    })

# ============================================================
# Aggregate Metrics
# ============================================================
df_metrics = pd.DataFrame(all_metrics)

print("\n=== TEST PERFORMANCE (MEAN ± STD) ===")
print(df_metrics.agg(["mean", "std"]))

# Save metrics for reuse
df_metrics.to_csv("classical_metrics.csv", index=False)
np.save("classical_metrics.npy", df_metrics.to_dict())      # <<< ADDED
np.save("classical_histories.npy", histories)

# Save final trained model
model.save("classical_model_final.keras")                   # <<< ADDED

# ============================================================
# Accuracy / Loss Curves (Final Run)
# ============================================================
h = histories[-1]

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(h["accuracy"], label="Train Accuracy")
plt.plot(h["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Classical BiLSTM Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(h["loss"], label="Train Loss")
plt.plot(h["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Classical BiLSTM Loss")
plt.legend()

plt.tight_layout()
plt.show()

# ============================================================
# Confusion Matrix
# ============================================================
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
plt.imshow(cm, cmap="Blues")
plt.title("Classical BiLSTM Confusion Matrix")
plt.colorbar()

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=12)

plt.xticks([0,1], ["Negative", "Positive"])
plt.yticks([0,1], ["Negative", "Positive"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

# ============================================================
# Inference Latency
# ============================================================
start = time.time()
_ = model.predict(X_test[:1000], verbose=0)
latency = (time.time() - start) / 1000
print(f"\nAvg inference latency per sample: {latency:.6f} seconds")
